# Práctica B: Rendimiento de Café con Datos Reales (Jaén, Cajamarca)

Este notebook implementa el caso de estudio **Caso B** con datos 100% reales y auditados:
- Rendimiento oficial de café extraído de compendios estadísticos del **INEI** y **MIDAGRI** (2010 a 2024).
- Índices de vegetación reales **MODIS MOD13Q1** (250m, 16 días) de Microsoft Planetary Computer.
- Clima diario real **ERA5** (Open-Meteo Archive, 2000–2024).
- Evaluación empírica de las 4 hipótesis científicas (H1: saturación de NDVI, H2: desacople fenológico, H3: ventaja de fusión multimodal en ventana expansiva, H4: invisibilidad de la roya 2012-2014 al satélite óptico).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json

df = pd.read_csv('../data/caso_2/cafe_jaen_campanias.csv')
print(f'Total campañas: {len(df)}')
con_rend = df.dropna(subset=['rendimiento_kg_ha'])
con_rend[['campania', 'rendimiento_qq_ha', 'superficie_cosechada_ha', 'produccion_t', 'ndvi_anual', 'lluvia_flor_mm', 'tmax_flor_c']]

## 1. Demostración de Saturación (H1): NDVI vs Rendimiento
Observemos cómo el NDVI promedio anual permanece prácticamente plano en ~0.74 (CV = 2.3%), desacoplado de las variaciones del rendimiento de cosecha.

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))

color = 'tab:blue'
ax1.set_xlabel('Campaña Agrícola')
ax1.set_ylabel('Rendimiento (qq/ha)', color=color)
line1 = ax1.plot(con_rend['campania'], con_rend['rendimiento_qq_ha'], 'o-', color=color, linewidth=2.5, label='Rendimiento Oficial INEI')
ax1.tick_params(axis='y', labelcolor=color)
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
color = 'tab:green'
ax2.set_ylabel('MODIS NDVI Anual (250m)', color=color)
line2 = ax2.plot(con_rend['campania'], con_rend['ndvi_anual'], 's--', color=color, linewidth=2, label='NDVI Anual (Satélite)')
ax2.set_ylim(0.5, 0.9)
ax2.tick_params(axis='y', labelcolor=color)

# Resaltar la crisis de la Roya 2012-2014
ax1.axvspan(2012, 2014, color='red', alpha=0.15, label='Crisis de la Roya Amarilla')

lines = line1 + line2
labels = [l.get_label() for l in lines] + ['Crisis de la Roya']
ax1.legend(loc='upper left')
plt.title('Caso B: La Trampa de la Saturación — Rendimiento Colapsa en 2014 mientras NDVI Permanece Verde')
plt.tight_layout()
plt.show()

## 2. Auditoría Científica de las 4 Hipótesis (H1, H2, H3, H4)

In [ ]:
with open('../data/caso_2/cafe_jaen_hipotesis.json', 'r', encoding='utf-8') as f:
    hip = json.load(f)
for h_key, h_val in hip.items():
    print(f'=== {h_key.upper()} ({h_val["estado"]}) ===')
    print(f'Premisa: {h_val["hipotesis"]}')
    print(f'Interpretación: {h_val["interpretacion"]}\n')

## 3. Resultados de Validación por Ventana Expansiva (Leave-Future-Out)

In [ ]:
df_res = pd.read_csv('../data/caso_2/cafe_jaen_resultados.csv')
print('Predicciones año a año sin contaminación futura:')
df_res